## Importing libraries

In [33]:
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import (RobertaTokenizer, RobertaForMaskedLM, 
                          DataCollatorForSeq2Seq, Seq2SeqTrainer, 
                          Seq2SeqTrainingArguments)
from transformers import RobertaForCausalLM, RobertaTokenizer

## Loading data

In [17]:
# Function combine java and C# datasets
def load_code_pairs(java_file, cs_file):
    java_code = open(java_file, 'r').readlines()
    cs_code = open(cs_file, 'r').readlines()
    return pd.DataFrame({
        'java_code': java_code,
        'cs_code': cs_code
    })

In [18]:
# Loading datasets
train_df = load_code_pairs('train.java-cs.txt.java', 'train.java-cs.txt.cs')
valid_df = load_code_pairs('valid.java-cs.txt.java', 'valid.java-cs.txt.cs')
test_df = load_code_pairs('test.java-cs.txt.java', 'test.java-cs.txt.cs')

In [19]:
# Function to reduce size of dataset
def reduce_dataset_size(df, percentage=0.1):
    return df.sample(frac=percentage, random_state=42)

In [20]:
# Reducing dataset size
train_df_10 = reduce_dataset_size(train_df, percentage=0.1)
valid_df_10 = reduce_dataset_size(valid_df, percentage=0.1)
test_df_10 = reduce_dataset_size(test_df, percentage=0.1)

In [21]:
# Convert to Datasets
train_dataset = Dataset.from_pandas(train_df_10)
valid_dataset = Dataset.from_pandas(valid_df_10)
test_dataset = Dataset.from_pandas(test_df_10)

## Model building using CodeBERT

In [22]:
# Tokenizer for CodeBERT
tokenizer = RobertaTokenizer.from_pretrained('microsoft/codebert-base')

In [23]:
# Tokenization function
def tokenize_function(examples):
    inputs = tokenizer(examples['java_code'], padding='max_length', truncation=True, max_length=512)
    labels = tokenizer(examples['cs_code'], padding='max_length', truncation=True, max_length=512)
    inputs['labels'] = labels['input_ids']
    return inputs

In [24]:
# Tokenized datasets
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_valid = valid_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/1030 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [25]:
# Loading CodeBERT model
model = RobertaForMaskedLM.from_pretrained('microsoft/codebert-base')

Some weights of RobertaForMaskedLM were not initialized from the model checkpoint at microsoft/codebert-base and are newly initialized: ['lm_head.bias', 'lm_head.decoder.bias', 'lm_head.dense.bias', 'lm_head.dense.weight', 'lm_head.layer_norm.bias', 'lm_head.layer_norm.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [26]:
# Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [27]:
# Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",             # Output directory for saving model checkpoints
    evaluation_strategy="steps",        # Evaluate after certain number of steps
    per_device_train_batch_size=4,      # Batch size for training
    per_device_eval_batch_size=4,       # Batch size for evaluation
    logging_dir="./logs",               # Directory for storing logs
    logging_steps=10,                   # Logging steps interval
    save_steps=50,                      # Save model checkpoint after every 50 steps
    eval_steps=50,                      # Evaluate every 50 steps
    save_total_limit=3,                 # Keep only the last 3 checkpoints
    num_train_epochs=1,                 # Number of training epochs
    predict_with_generate=True          # Use generation during prediction
)


/Users/himanshuthakur/anaconda3/lib/python3.11/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [28]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    tokenizer=tokenizer,
    data_collator=data_collator
)

In [29]:
# Trainning model
trainer.train()

Step,Training Loss,Validation Loss
50,0.835600,1.138053
100,1.075300,1.036262
150,0.785600,0.985292
200,0.817900,0.957429
250,0.780000,0.943890


TrainOutput(global_step=258, training_loss=1.3879317294719606, metrics={'train_runtime': 4222.9794, 'train_samples_per_second': 0.244, 'train_steps_per_second': 0.061, 'total_flos': 271163427194880.0, 'train_loss': 1.3879317294719606, 'epoch': 1.0})

In [30]:
# Model performance on validation dataset
metrics = trainer.evaluate()
print("Evaluation metrics on reduced dataset:", metrics)

Evaluation metrics on reduced dataset: {'eval_loss': 0.943618893623352, 'eval_runtime': 38.7334, 'eval_samples_per_second': 1.291, 'eval_steps_per_second': 0.336, 'epoch': 1.0}


In [34]:
# Load the causal LM model for sequence generation
model = RobertaForCausalLM.from_pretrained('microsoft/codebert-base')
tokenizer = RobertaTokenizer.from_pretrained('microsoft/codebert-base')

If you want to use `RobertaLMHeadModel` as a standalone, add `is_decoder=True.`
Some weights of RobertaForCausalLM were not initialized from the model checkpoint at microsoft/codebert-base and are newly initialized: ['lm_head.bias', 'lm_head.decoder.bias', 'lm_head.dense.bias', 'lm_head.dense.weight', 'lm_head.layer_norm.bias', 'lm_head.layer_norm.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Users/himanshuthakur/anaconda3/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


## Java to C# code translation

In [35]:
# Function to translate Java code to C#
def translate_java_to_cs(java_code):
    inputs = tokenizer(java_code, return_tensors="pt", padding=True, truncation=True).to(model.device)
    translated_outputs = model.generate(**inputs, max_length=512)
    return tokenizer.decode(translated_outputs[0], skip_special_tokens=True)

In [37]:
# Example Java and corresponding C# code from the test dataset (assuming 'java_code' and 'cs_code' columns exist)
example_java_code = test_df_10['java_code'].iloc[0]

# Translate the Java code to C#
translated_cs_code = translate_java_to_cs(example_java_code)

# Print the original Java code, adjacent C# code, and translated C# code
print(f"Original Java Code:\n{example_java_code}")
print(f"Translated C# Code:\n{translated_cs_code}")

Original Java Code:
public void add(int location, E object) {if (location >= 0 && location <= size) {Link<E> link = voidLink;if (location < (size / 2)) {for (int i = 0; i <= location; i++) {link = link.next;}} else {for (int i = size; i > location; i--) {link = link.previous;}}Link<E> previous = link.previous;Link<E> newLink = new Link<E>(object, previous, link);previous.next = newLink;link.previous = newLink;size++;modCount++;} else {throw new IndexOutOfBoundsException();}}

Translated C# Code:
public void add(int location, E object) {if (location >= 0 && location <= size) {Link<E> link = voidLink;if (location < (size / 2)) {for (int i = 0; i <= location; i++) {link = link.next;}} else {for (int i = size; i > location; i--) {link = link.previous;}}Link<E> previous = link.previous;Link<E> newLink = new Link<E>(object, previous, link);previous.next = newLink;link.previous = newLink;size++;modCount++;} else {throw new IndexOutOfBoundsException();}}
arily farmland� farmland childbirtharil